# Sala Chaturamuk Phaichit — 3D Gaussian Splatting

Stage 2 of the reconstruction comparison. Stage 1 (COLMAP structure-from-motion) already ran locally and registered all 146 photographs of `salathai_version2` into a single model at 0.58 px mean reprojection error; this notebook consumes those camera poses.

**This is a comparison, not a replacement.** The project's contribution is image-based rendering that synthesizes novel views *without* recovering geometry. This notebook answers the adjacent question: what do you get, and what does it cost, if you do recover geometry from the same photographs?

## Before you start

**Set the runtime to a GPU:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**. Nothing here works on CPU.

You will need `sala_v2_colmap.zip` (105 MB), which contains `images/` and `sparse/0/`. Upload it to your Google Drive first — mounting Drive is far more reliable than a direct upload, which restarts from zero if the connection drops.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'

## 2. Get the data

Put `sala_v2_colmap.zip` in the top level of your Drive ("My Drive"), then run this. If you would rather upload directly, replace this cell with `from google.colab import files; files.upload()` — but expect it to be slower and less forgiving.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp '/content/drive/My Drive/sala_v2_colmap.zip' /content/
!cd /content && unzip -q sala_v2_colmap.zip

# Expect 146 images and a sparse model of three .bin files.
!echo "images: $(ls /content/sala_v2_colmap/images | wc -l)"
!ls -la /content/sala_v2_colmap/sparse/0/

## 3. Install 3D Gaussian Splatting

The two submodules are CUDA extensions and compile against the runtime's toolkit, so this cell takes several minutes. It is the slowest setup step.

*If the build fails*, the usual cause is a CUDA/PyTorch version mismatch in a fresh Colab image. The fallback that avoids compiling anything is `nerfstudio`'s `splatfacto` — see the last section.

In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
%cd /content/gaussian-splatting
!pip install -q plyfile
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

## 4. Train

30 000 iterations is the paper's default and takes roughly 30–50 minutes for 146 images on a T4. `--save_iterations` writes intermediate models so a disconnect does not cost you everything.

Colab disconnects idle sessions — keep the tab visible while this runs.

In [ ]:
%cd /content/gaussian-splatting
!python train.py \
  -s /content/sala_v2_colmap \
  -m /content/output/sala_v2 \
  --iterations 30000 \
  --save_iterations 7000 15000 30000 \
  --eval

## 5. Measure it

`--eval` above held out every 8th photograph, so these PSNR/SSIM/LPIPS figures are scored against real photographs the model never saw.

This matters for the write-up: it is the *same kind of held-out measurement* the IBR pipeline's `src/evaluate.py` performs, so the two approaches can be compared on the same footing rather than by eye.

In [ ]:
%cd /content/gaussian-splatting
!python render.py -m /content/output/sala_v2
!python metrics.py -m /content/output/sala_v2

## 6. Save the result back to Drive

`point_cloud.ply` **is** the 3D model — a few hundred thousand oriented gaussians, each with position, covariance, opacity and view-dependent colour. Drag it onto <https://superspl.at/editor> or <https://playcanvas.com/supersplat/editor> to fly around it in a browser, which is the strongest way to show it live.

In [ ]:
import shutil, os

PLY = '/content/output/sala_v2/point_cloud/iteration_30000/point_cloud.ply'
print('exists:', os.path.isfile(PLY), '|',
      round(os.path.getsize(PLY) / 1e6, 1) if os.path.isfile(PLY) else 0, 'MB')

shutil.copy(PLY, '/content/drive/My Drive/sala_v2_gaussians.ply')

# The rendered held-out comparisons are worth keeping as paper figures:
# renders/ is what the model produced, gt/ is the real photograph.
shutil.make_archive('/content/drive/My Drive/sala_v2_renders', 'zip',
                    '/content/output/sala_v2/test')
print('saved to Drive')

## What to expect, honestly

Both captures are **single-height rings** — every photograph was taken from roughly eye level while walking around the pavilion. So:

- **The sides should reconstruct well.** Dense coverage, 2.4° mean angular spacing, good parallax.
- **The roof will not.** Nothing ever looked down on it, so expect noise or a hole above the eaves. This is a limitation of the capture, not of the method, and is worth reporting rather than cropping out of the demo.
- **Viewpoints far from the ring will degrade.** Gaussian splatting interpolates confidently near the training cameras and invents detail away from them.

The measured camera path (`recon/salathai_version2/camera_path.json`) puts the walk at **342°**, not a full revolution, ending 2.21 units from where it began where a normal step is 0.19 — so there is one wider gap in the ring. Expect the reconstruction to be weakest there.

## Fallback: nerfstudio

If the CUDA extensions above refuse to build, this compiles nothing and its `splatfacto` method is the same family of technique:

```bash
!pip install nerfstudio
!ns-train splatfacto --data /content/sala_v2_colmap
```

`ns-train nerfacto` trains a NeRF instead — literally a neural network, and the more textbook answer if that phrasing matters for the assignment.